In [1]:
# Load the extension that allows us to compile CUDA code in python notebooks
# Documentation is here: https://nvcc4jupyter.readthedocs.io/en/latest/
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-jwmwnxph
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-jwmwnxph
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 28f872a2f99a1b201bcd0db14fdbc5a496b9bfd7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nvcc4jupyter: filename=nvcc4jupyter-1.2.1-py3-none-any.whl size=10743 sha256=3a570d3ab74e9051a615aa78d54722c78a69a0c83a1d407fccde14c37a30e96c
  Stored in directory: /tmp/pip-ephem-wheel-cache-0u46w5qn/wheels/a8/b9/18/23f8ef71ceb0f63297dd1903aedd067e6243a68ea756d6feea
Successfully built nvcc4jupyter
Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpkxbksdcx".


In [2]:
%%cuda_group_save -g "source" -n "data_types.h"
/**
 * A collection of commonly used data types throughout this project.
 */
#pragma once

#include <stdint.h> // uint32_t

using element_t = uint32_t;

In [3]:
%%cuda_group_save -g "source" -n "cuda_common.h"
/**
 * Standard macros that can be useful for error checking.
 * https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__ERROR.html
 */
#pragma once

#include <cuda.h>

#define CUDA_CALL(exp)                                       \
    do {                                                     \
        cudaError res = (exp);                               \
        if(res != cudaSuccess) {                             \
            printf("Error at %s:%d\n %s\n",                  \
                __FILE__,__LINE__, cudaGetErrorString(res)); \
           exit(EXIT_FAILURE);                               \
        }                                                    \
    } while(0)

#define CHECK_ERROR(msg)                                             \
    do {                                                             \
        cudaError_t err = cudaGetLastError();                        \
        if(cudaSuccess != err) {                                     \
            printf("Error (%s) at %s:%d\n %s\n",                     \
                (msg), __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE);                                      \
        }                                                            \
    } while (0)

In [4]:
%%cuda_group_save -g "source" -n "data_generator.h"
/**
 * Functions for generating random input data with a fixed seed
 */
#pragma once

#include <random>  // for std::mt19937, std::uniform_int_distribution
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a1 {

/**
 * Generates and returns a vector of random uniform data of a given length, n,
 * for any integral type. Input range will be [0, 2n].
 */
template < typename T >
std::vector< T > generate_uniform( std::size_t n )
{
    // for details of random number generation, see:
    // https://en.cppreference.com/w/cpp/numeric/random/uniform_int_distribution
    std::size_t random_seed = 20240916;  // use magic seed
    std::mt19937 rng( random_seed );     // use mersenne twister generator
    std::uniform_int_distribution<> distrib(0, 2 * n);

    std::vector< T > random_data( n ); // init array
    std::generate( std::begin( random_data )
                 , std::end  ( random_data )
                 , [ &rng, &distrib ](){ return static_cast< T >( distrib( rng ) ); });

    return random_data;
}

} // namespace a1
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "algorithm_choices.h"
#pragma once

#include <cstddef>
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a1 {

/**
 * What every run (CPU or GPU) hands back: the transformed array and the
 * wall time of the timed region in nanoseconds.
 */
struct run_result
{
    std::vector< element_t > output;
    long long ns;
};

namespace cpu {

run_result run_cpu_baseline( std::vector< element_t > data, std::size_t switch_at, std::size_t n );

} // namespace cpu


namespace gpu {

/// Which sorting network / algorithm to use on the device.
enum class sort_algo { bitonic, radix };

/// Which memory strategy to use. `automatic` uses shared memory once n >= 1024.
enum class mem_mode { global_only, shared, automatic };

/// Size at (and above) which `mem_mode::automatic` switches to shared memory.
constexpr std::size_t shared_memory_threshold = 1024;

run_result run_gpu_soln( std::vector< element_t > data, std::size_t switch_at, std::size_t n,
                         sort_algo algo = sort_algo::bitonic,
                         mem_mode  mode = mem_mode::automatic );

} // namespace gpu
} // namespace a1
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "cpu_baseline.cu"
/**
 * CPU methods that the GPU should outperform.
 */

#include "algorithm_choices.h"

#include <algorithm> // std::sort()
#include <chrono>    // for timing
#include <iostream>  // std::cout, std::endl

namespace csc485b {
namespace a1      {
namespace cpu     {

/**
 * Simple solution that just sorts the whole array with a built-in sort
 * function and then resorts the last portion in the opposing order with
 * a second call to that same built-in sort function.
 */
void opposing_sort( element_t * data, std::size_t invert_at_pos, std::size_t num_elements )
{
    std::sort( data, data + num_elements, std::less< element_t >{} );
    std::sort( data + invert_at_pos, data + num_elements, std::greater< element_t >{} );
}

/**
 * Run the single-threaded CPU baseline that students are supposed to outperform
 * in order to obtain higher grades on this assignment. Times the execution and
 * returns that "wall time" together with the result. Note that the function
 * takes the input by value so as to not perturb the original data in place.
 */
run_result run_cpu_baseline( std::vector< element_t > data, std::size_t switch_at, std::size_t n )
{
    auto const cpu_start = std::chrono::high_resolution_clock::now();
    opposing_sort( data.data(), switch_at, n );
    auto const cpu_end = std::chrono::high_resolution_clock::now();

    auto const ns = std::chrono::duration_cast< std::chrono::nanoseconds >( cpu_end - cpu_start ).count();
    return { std::move( data ), static_cast< long long >( ns ) };
}

} // namespace cpu
} // namespace a1
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "gpu_solution.cu"
/**
 * GPU solutions for the opposing sort:
 *
 *   - A bitonic sorting network ("bitonic near-sort"), in a global-memory-only
 *     variant and a shared-memory variant.
 *   - A massively parallel LSD radix sort, also in a global-memory-only
 *     variant and a shared-memory variant, used to benchmark the bitonic sort.
 *
 * Both algorithms sort the whole array ascending and finish with one parallel
 * pass that reverses [invert_at_pos, n), which turns the largest n/4 elements
 * into a descending run. That matches the CPU baseline exactly.
 */

#include "algorithm_choices.h"

#include <chrono>    // for timing
#include <iostream>  // std::cout, std::endl
#include <utility>   // std::swap (host only)
#include <vector>

#include "cuda_common.h"

namespace csc485b {
namespace a1      {
namespace gpu     {

namespace {

// ---------------------------------------------------------------------------
// Configuration
// ---------------------------------------------------------------------------

/// Threads per block for every kernel (max block size on a T4).
constexpr std::size_t THREADS = 1024;

/// Bitonic: each thread owns one compare-exchange, i.e. two elements.
constexpr std::size_t BITONIC_TILE = 2 * THREADS;

/// Radix (shared): one element per thread per tile.
constexpr std::size_t RADIX_TILE = THREADS;

/// Radix digit width. 4 bits -> 16 buckets, 8 passes for 32-bit keys.
constexpr unsigned RADIX_BITS    = 4;
constexpr unsigned RADIX_BUCKETS = 1u << RADIX_BITS;
constexpr unsigned RADIX_PASSES  = ( 8 * sizeof( element_t ) ) / RADIX_BITS;
static_assert( RADIX_PASSES % 2 == 0, "ping-pong must end in the original buffer" );

/// Radix (global): each thread serially processes a contiguous chunk.
constexpr std::size_t RADIX_ITEMS_PER_THREAD = 32;

/// Padding value: sorts after every real key.
constexpr element_t PAD = static_cast< element_t >( ~0u );

std::size_t blocks_for( std::size_t work, std::size_t per_block = THREADS )
{
    return ( work + per_block - 1 ) / per_block;
}

std::size_t next_pow2( std::size_t x )
{
    std::size_t p = 1;
    while( p < x ) p <<= 1;
    return p;
}

} // anonymous namespace

/**
 * The CPU baseline benefits from warm caches because the data was generated on
 * the CPU. Run the data through the GPU once with some arbitrary logic to
 * ensure that the GPU cache is warm too and the comparison is more fair.
 */
__global__
void warm_the_gpu( element_t * data, std::size_t invert_at_pos, std::size_t num_elements )
{
    int const th_id = blockIdx.x * blockDim.x + threadIdx.x;

    // We know this will never be true, because of the data generator logic,
    // but I doubt that the compiler will figure it out. Thus every element
    // should be read, but none of them should be modified.
    if( th_id < num_elements && data[ th_id ] > num_elements * 100 )
    {
        ++data[ th_id ]; // should not be possible.
    }
}

// ===========================================================================
// Shared helpers
// ===========================================================================

__device__ __forceinline__
void compare_exchange( element_t & a, element_t & b, bool ascending )
{
    if( ( a > b ) == ascending )
    {
        element_t const t = a; a = b; b = t;
    }
}

/**
 * Reverses data[begin, end) in place. This is what turns the tail of the
 * ascending result into the descending run. One thread per swapped pair.
 */
__global__
void reverse_range( element_t * data, std::size_t begin, std::size_t end )
{
    std::size_t const t   = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x;
    std::size_t const len = end - begin;
    if( t < len / 2 )
    {
        element_t const a = data[ begin + t ];
        data[ begin + t ]   = data[ end - 1 - t ];
        data[ end - 1 - t ] = a;
    }
}

// ===========================================================================
// Bitonic sort
//
// Network over N = 2^p elements (input padded with PAD). Stage k doubles the
// length of the sorted bitonic runs; step j compares elements j apart. The
// pair handled by "thread" t is
//     lo = 2j * floor(t / j) + (t mod j),   hi = lo + j
// and the comparison is ascending iff bit k of lo is clear. In the final stage
// (k = N) every comparison is ascending, so the result is fully ascending.
// ===========================================================================

__device__ __forceinline__
std::size_t bitonic_lo( std::size_t t, std::size_t j )
{
    // j is a power of two: (t / j) * 2j + t % j, with masks instead of div/mod
    return ( ( t & ~( j - 1 ) ) << 1 ) | ( t & ( j - 1 ) );
}

/**
 * Global-memory-only version: one launch per (k, j) step, every compare reads
 * and writes DRAM. The kernel boundary acts as the grid-wide barrier.
 */
__global__
void bitonic_step_global( element_t * data, std::size_t N, std::size_t k, std::size_t j )
{
    std::size_t const t = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x;
    if( t >= N / 2 ) return;

    std::size_t const lo = bitonic_lo( t, j );
    std::size_t const hi = lo + j;

    element_t a = data[ lo ];
    element_t b = data[ hi ];
    compare_exchange( a, b, ( lo & k ) == 0 );
    data[ lo ] = a;
    data[ hi ] = b;
}

/**
 * Shared-memory version, phase 1: every block loads a 2048-element tile and
 * runs ALL stages k = 2 .. 2048 on it in shared memory (66 steps, one global
 * read and one global write per element). Directions are taken from the global
 * index so neighbouring tiles come out in opposite orders, ready for merging.
 */
__global__
void bitonic_sort_tiles_shared( element_t * data )
{
    __shared__ element_t s[ BITONIC_TILE ];

    unsigned    const tid  = threadIdx.x;
    std::size_t const base = blockIdx.x * BITONIC_TILE;

    s[ tid ]           = data[ base + tid ];
    s[ tid + THREADS ] = data[ base + tid + THREADS ];
    __syncthreads();

    for( std::size_t k = 2; k <= BITONIC_TILE; k <<= 1 )
    {
        for( std::size_t j = k >> 1; j > 0; j >>= 1 )
        {
            std::size_t const lo = bitonic_lo( tid, j );
            compare_exchange( s[ lo ], s[ lo + j ], ( ( base + lo ) & k ) == 0 );
            __syncthreads();
        }
    }

    data[ base + tid ]           = s[ tid ];
    data[ base + tid + THREADS ] = s[ tid + THREADS ];
}

/**
 * Shared-memory version, phase 2: for a stage k > 2048, once the stride drops
 * to j <= 1024 every remaining compare stays inside one tile, so the whole
 * tail of the stage (11 steps) is done in shared memory by one launch.
 */
__global__
void bitonic_merge_tiles_shared( element_t * data, std::size_t k )
{
    __shared__ element_t s[ BITONIC_TILE ];

    unsigned    const tid  = threadIdx.x;
    std::size_t const base = blockIdx.x * BITONIC_TILE;

    // k > tile size, so the direction is the same for the whole tile.
    bool const ascending = ( base & k ) == 0;

    s[ tid ]           = data[ base + tid ];
    s[ tid + THREADS ] = data[ base + tid + THREADS ];
    __syncthreads();

    for( std::size_t j = THREADS; j > 0; j >>= 1 )
    {
        std::size_t const lo = bitonic_lo( tid, j );
        compare_exchange( s[ lo ], s[ lo + j ], ascending );
        __syncthreads();
    }

    data[ base + tid ]           = s[ tid ];
    data[ base + tid + THREADS ] = s[ tid + THREADS ];
}

/**
 * Host driver for the bitonic network over N (a power of two) elements.
 * With shared memory, N must also be a multiple of BITONIC_TILE.
 */
void bitonic_sort_ascending( element_t * d_data, std::size_t N, bool use_shared )
{
    if( N < 2 ) return;
    std::size_t const step_blocks = blocks_for( N / 2 );

    if( !use_shared )
    {
        for( std::size_t k = 2; k <= N; k <<= 1 )
            for( std::size_t j = k >> 1; j > 0; j >>= 1 )
                bitonic_step_global<<< step_blocks, THREADS >>>( d_data, N, k, j );
        return;
    }

    std::size_t const tiles = N / BITONIC_TILE;
    bitonic_sort_tiles_shared<<< tiles, THREADS >>>( d_data );

    for( std::size_t k = 2 * BITONIC_TILE; k <= N; k <<= 1 )
    {
        // Strides that cross tile boundaries need a grid-wide barrier -> global
        for( std::size_t j = k >> 1; j >= BITONIC_TILE; j >>= 1 )
            bitonic_step_global<<< step_blocks, THREADS >>>( d_data, N, k, j );
        // Strides 1024 .. 1 are tile-local -> one shared-memory launch
        bitonic_merge_tiles_shared<<< tiles, THREADS >>>( d_data, k );
    }
}

// ===========================================================================
// Radix sort (LSD, 4-bit digits, 8 passes)
//
// Each pass is: per-tile digit histogram -> global exclusive scan of the
// histograms (laid out digit-major so the scan yields every tile's output
// offset per digit) -> stable scatter. Passes ping-pong between two buffers.
// ===========================================================================

__device__ __forceinline__
unsigned digit_of( element_t key, unsigned shift )
{
    return ( key >> shift ) & ( RADIX_BUCKETS - 1 );
}

// ---------- shared-memory variant ------------------------------------------

/**
 * Block-wide inclusive scan (Hillis-Steele, double-buffered in shared memory)
 * of one value per thread. Every thread of the block must call it. `buf` must
 * hold 2 * THREADS entries. `total` receives the sum over the block.
 */
__device__
unsigned block_inclusive_scan( unsigned value, unsigned * buf, unsigned & total )
{
    unsigned const tid = threadIdx.x;
    unsigned * in  = buf;
    unsigned * out = buf + THREADS;

    in[ tid ] = value;
    __syncthreads();

    for( unsigned offset = 1; offset < THREADS; offset <<= 1 )
    {
        unsigned x = in[ tid ];
        if( tid >= offset ) x += in[ tid - offset ];
        out[ tid ] = x;
        __syncthreads();
        unsigned * const tmp = in; in = out; out = tmp;
    }

    unsigned const result = in[ tid ];
    total = in[ THREADS - 1 ];
    __syncthreads(); // buffer may be reused by the caller right away
    return result;
}

/**
 * Sorts each 1024-key tile by the current digit in shared memory (four stable
 * 1-bit splits), writes the locally sorted tile back in place, and records for
 * each digit how many keys the tile has (counts, digit-major) and where that
 * digit's run starts inside the tile (tile_starts, tile-major).
 *
 * Locally sorting first makes the later scatter write long contiguous runs,
 * i.e. mostly coalesced stores.
 */
__global__
void radix_tile_sort_shared( element_t * keys, std::size_t n, unsigned shift,
                             unsigned * counts, unsigned * tile_starts )
{
    __shared__ element_t s_keys [ RADIX_TILE ];
    __shared__ unsigned  s_scan [ 2 * THREADS ];
    __shared__ unsigned  s_start[ RADIX_BUCKETS ];
    __shared__ unsigned  s_end  [ RADIX_BUCKETS ];

    unsigned    const tid   = threadIdx.x;
    std::size_t const base  = blockIdx.x * RADIX_TILE;
    std::size_t const valid = ( n - base < RADIX_TILE ) ? n - base : RADIX_TILE;

    // Out-of-range slots get PAD (all ones). PAD lands in the "1" side of every
    // split and the splits are stable, so padding always ends up at the tail.
    element_t key = ( tid < valid ) ? keys[ base + tid ] : PAD;

    if( tid < RADIX_BUCKETS ) { s_start[ tid ] = 0; s_end[ tid ] = 0; }

    for( unsigned b = 0; b < RADIX_BITS; ++b )
    {
        unsigned const bit = ( key >> ( shift + b ) ) & 1u;
        unsigned total_zeros;
        unsigned const zeros_incl   = block_inclusive_scan( 1u - bit, s_scan, total_zeros );
        unsigned const zeros_before = zeros_incl - ( 1u - bit );
        unsigned const dest = bit ? total_zeros + ( tid - zeros_before ) // ones before me
                                  : zeros_before;
        s_keys[ dest ] = key;
        __syncthreads();
        key = s_keys[ tid ];
        __syncthreads();
    }

    // Find each digit's run [start, end) inside the sorted tile.
    unsigned const d = digit_of( key, shift );
    if( tid < valid )
    {
        if( tid == 0         || digit_of( s_keys[ tid - 1 ], shift ) != d ) s_start[ d ] = tid;
        if( tid == valid - 1 || digit_of( s_keys[ tid + 1 ], shift ) != d ) s_end  [ d ] = tid + 1;
    }
    __syncthreads();

    if( tid < valid ) keys[ base + tid ] = key;

    if( tid < RADIX_BUCKETS )
    {
        counts     [ tid * gridDim.x + blockIdx.x ]     = s_end[ tid ] - s_start[ tid ];
        tile_starts[ blockIdx.x * RADIX_BUCKETS + tid ] = s_start[ tid ];
    }
}

/**
 * Stable scatter: key at local position i with digit d goes to
 *   global_offset[d][tile] + (i - start of d's run in this tile).
 */
__global__
void radix_scatter_shared( element_t const * in, element_t * out, std::size_t n, unsigned shift,
                           unsigned const * offsets, unsigned const * tile_starts )
{
    __shared__ unsigned s_offset[ RADIX_BUCKETS ];
    __shared__ unsigned s_start [ RADIX_BUCKETS ];

    unsigned    const tid   = threadIdx.x;
    std::size_t const base  = blockIdx.x * RADIX_TILE;
    std::size_t const valid = ( n - base < RADIX_TILE ) ? n - base : RADIX_TILE;

    if( tid < RADIX_BUCKETS )
    {
        s_offset[ tid ] = offsets    [ tid * gridDim.x + blockIdx.x ];
        s_start [ tid ] = tile_starts[ blockIdx.x * RADIX_BUCKETS + tid ];
    }
    __syncthreads();

    if( tid < valid )
    {
        element_t const key = in[ base + tid ];
        unsigned  const d   = digit_of( key, shift );
        out[ s_offset[ d ] + ( tid - s_start[ d ] ) ] = key;
    }
}

/// Exclusive scan of each 1024-element segment in shared memory; segment totals -> block_sums.
__global__
void scan_blocks_shared( unsigned * data, std::size_t m, unsigned * block_sums )
{
    __shared__ unsigned s_scan[ 2 * THREADS ];

    std::size_t const i = blockIdx.x * THREADS + threadIdx.x;
    unsigned const v = ( i < m ) ? data[ i ] : 0u;

    unsigned total;
    unsigned const incl = block_inclusive_scan( v, s_scan, total );

    if( i < m ) data[ i ] = incl - v;
    if( threadIdx.x == 0 && block_sums != nullptr ) block_sums[ blockIdx.x ] = total;
}

__global__
void add_block_offsets( unsigned * data, std::size_t m, unsigned const * block_offsets )
{
    std::size_t const i = blockIdx.x * THREADS + threadIdx.x;
    if( i < m ) data[ i ] += block_offsets[ blockIdx.x ];
}

/// Scratch buffers for the recursive scan, allocated before timing starts.
struct scan_levels
{
    std::vector< unsigned * > sums;
};

void alloc_scan_levels( std::size_t m, scan_levels & levels )
{
    while( m > THREADS )
    {
        std::size_t const b = blocks_for( m );
        unsigned * p;
        cudaMalloc( (void**) & p, sizeof( unsigned ) * b );
        levels.sums.push_back( p );
        m = b;
    }
}

void free_scan_levels( scan_levels & levels )
{
    for( auto p : levels.sums ) cudaFree( p );
    levels.sums.clear();
}

/// In-place exclusive scan of m values (scan segments, scan the sums, add back).
void exclusive_scan_shared( unsigned * d_data, std::size_t m, scan_levels & levels, std::size_t level = 0 )
{
    std::size_t const b = blocks_for( m );
    unsigned * const sums = ( b > 1 ) ? levels.sums[ level ] : nullptr;

    scan_blocks_shared<<< b, THREADS >>>( d_data, m, sums );
    if( b > 1 )
    {
        exclusive_scan_shared( sums, b, levels, level + 1 );
        add_block_offsets<<< b, THREADS >>>( d_data, m, sums );
    }
}

// ---------- global-memory-only variant -------------------------------------

/**
 * Each thread counts the digits of its own contiguous chunk of keys in
 * registers. Counts are stored digit-major: counts[d * num_threads + t].
 */
__global__
void radix_histogram_global( element_t const * keys, std::size_t n, unsigned shift,
                             unsigned * counts, std::size_t num_threads )
{
    std::size_t const t = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x;
    if( t >= num_threads ) return;

    unsigned c[ RADIX_BUCKETS ];
    for( unsigned d = 0; d < RADIX_BUCKETS; ++d ) c[ d ] = 0;

    std::size_t const begin = t * RADIX_ITEMS_PER_THREAD;
    std::size_t const end   = ( begin + RADIX_ITEMS_PER_THREAD < n ) ? begin + RADIX_ITEMS_PER_THREAD : n;
    for( std::size_t i = begin; i < end; ++i ) ++c[ digit_of( keys[ i ], shift ) ];

    for( unsigned d = 0; d < RADIX_BUCKETS; ++d ) counts[ d * num_threads + t ] = c[ d ];
}

/// Each thread walks its chunk in order, so the scatter is stable.
__global__
void radix_scatter_global( element_t const * in, element_t * out, std::size_t n, unsigned shift,
                           unsigned const * offsets, std::size_t num_threads )
{
    std::size_t const t = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x;
    if( t >= num_threads ) return;

    unsigned o[ RADIX_BUCKETS ];
    for( unsigned d = 0; d < RADIX_BUCKETS; ++d ) o[ d ] = offsets[ d * num_threads + t ];

    std::size_t const begin = t * RADIX_ITEMS_PER_THREAD;
    std::size_t const end   = ( begin + RADIX_ITEMS_PER_THREAD < n ) ? begin + RADIX_ITEMS_PER_THREAD : n;
    for( std::size_t i = begin; i < end; ++i )
    {
        element_t const key = in[ i ];
        out[ o[ digit_of( key, shift ) ]++ ] = key;
    }
}

/// out[i] = in[i-1] (0 for i = 0): turns an inclusive scan into an exclusive one.
__global__
void shift_right_global( unsigned const * in, unsigned * out, std::size_t m )
{
    std::size_t const i = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x;
    if( i < m ) out[ i ] = ( i == 0 ) ? 0u : in[ i - 1 ];
}

/// One Hillis-Steele step entirely in global memory.
__global__
void hillis_steele_step_global( unsigned const * in, unsigned * out, std::size_t m, std::size_t offset )
{
    std::size_t const i = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x;
    if( i < m ) out[ i ] = in[ i ] + ( ( i >= offset ) ? in[ i - offset ] : 0u );
}

/// Exclusive scan of `counts` using only global memory. Clobbers both buffers;
/// returns whichever one holds the result.
unsigned * exclusive_scan_global( unsigned * counts, unsigned * scratch, std::size_t m )
{
    std::size_t const b = blocks_for( m );
    shift_right_global<<< b, THREADS >>>( counts, scratch, m );

    unsigned * in  = scratch;
    unsigned * out = counts;
    for( std::size_t offset = 1; offset < m; offset <<= 1 )
    {
        hillis_steele_step_global<<< b, THREADS >>>( in, out, m, offset );
        std::swap( in, out );
    }
    return in;
}

// ---------- radix driver ----------------------------------------------------

struct radix_scratch
{
    element_t * tmp         = nullptr; // ping-pong partner for the keys
    unsigned  * counts      = nullptr; // per-(digit, tile/thread) counts
    unsigned  * aux         = nullptr; // tile_starts (shared) or scan scratch (global)
    std::size_t units       = 0;       // tiles (shared) or threads (global)
    scan_levels levels;
};

void alloc_radix_scratch( std::size_t n, bool use_shared, radix_scratch & s )
{
    s.units = use_shared ? blocks_for( n, RADIX_TILE ) : blocks_for( n, RADIX_ITEMS_PER_THREAD );
    std::size_t const m = RADIX_BUCKETS * s.units;
    cudaMalloc( (void**) & s.tmp,    sizeof( element_t ) * n );
    cudaMalloc( (void**) & s.counts, sizeof( unsigned )  * m );
    cudaMalloc( (void**) & s.aux,    sizeof( unsigned )  * m );
    if( use_shared ) alloc_scan_levels( m, s.levels );
    CHECK_ERROR( "Allocating radix scratch" );
}

void free_radix_scratch( radix_scratch & s )
{
    cudaFree( s.tmp ); cudaFree( s.counts ); cudaFree( s.aux );
    free_scan_levels( s.levels );
}

void radix_sort_ascending( element_t * d_keys, std::size_t n, bool use_shared, radix_scratch & s )
{
    if( n < 2 ) return;
    std::size_t const m = RADIX_BUCKETS * s.units;

    element_t * in  = d_keys;
    element_t * out = s.tmp;

    for( unsigned pass = 0; pass < RADIX_PASSES; ++pass )
    {
        unsigned const shift = pass * RADIX_BITS;

        if( use_shared )
        {
            radix_tile_sort_shared<<< s.units, THREADS >>>( in, n, shift, s.counts, s.aux );
            exclusive_scan_shared( s.counts, m, s.levels );
            radix_scatter_shared<<< s.units, THREADS >>>( in, out, n, shift, s.counts, s.aux );
        }
        else
        {
            std::size_t const b = blocks_for( s.units );
            radix_histogram_global<<< b, THREADS >>>( in, n, shift, s.counts, s.units );
            unsigned const * offsets = exclusive_scan_global( s.counts, s.aux, m );
            radix_scatter_global<<< b, THREADS >>>( in, out, n, shift, offsets, s.units );
        }

        element_t * const t = in; in = out; out = t;
    }
    // RADIX_PASSES is even, so the sorted keys are back in d_keys.
}

// ===========================================================================
// Entry point
// ===========================================================================

/**
 * Performs all the logic of allocating device vectors and copying host/input
 * vectors to the device. Times the sort with wall time (synchronising before
 * the clock stops, since kernel launches are asynchronous), but excludes set
 * up and tear down costs such as mallocs, frees, and memcpies.
 */
run_result run_gpu_soln( std::vector< element_t > data, std::size_t switch_at, std::size_t n,
                         sort_algo algo, mem_mode mode )
{
    if( n == 0 ) return { std::move( data ), 0 };

    bool const use_shared = mode == mem_mode::shared
                         || ( mode == mem_mode::automatic && n >= shared_memory_threshold );

    // Bitonic networks need a power-of-two length (and >= one tile when using
    // shared memory). The padding is PAD, which sorts to the very end.
    std::size_t alloc_n = n;
    if( algo == sort_algo::bitonic )
    {
        alloc_n = next_pow2( n );
        if( use_shared && alloc_n < BITONIC_TILE ) alloc_n = BITONIC_TILE;
    }

    element_t * d_data;
    cudaMalloc( (void**) & d_data, sizeof( element_t ) * alloc_n );
    CHECK_ERROR( "Allocating input array on device" );
    cudaMemset( d_data, 0xFF, sizeof( element_t ) * alloc_n );            // PAD everywhere
    cudaMemcpy( d_data, data.data(), sizeof( element_t ) * n, cudaMemcpyHostToDevice );
    CHECK_ERROR( "Copying input array to device" );

    radix_scratch scratch;
    if( algo == sort_algo::radix ) alloc_radix_scratch( n, use_shared, scratch );

    // Warm the cache on the GPU for a more fair comparison
    warm_the_gpu<<< blocks_for( n ), THREADS >>>( d_data, switch_at, n );
    cudaDeviceSynchronize();

    auto const kernel_start = std::chrono::high_resolution_clock::now();

    if( algo == sort_algo::bitonic ) bitonic_sort_ascending( d_data, alloc_n, use_shared );
    else                             radix_sort_ascending  ( d_data, n, use_shared, scratch );

    if( switch_at < n && ( n - switch_at ) >= 2 )
        reverse_range<<< blocks_for( ( n - switch_at ) / 2 ), THREADS >>>( d_data, switch_at, n );

    cudaDeviceSynchronize();
    auto const kernel_end = std::chrono::high_resolution_clock::now();
    CHECK_ERROR( "Executing kernels on device" );

    cudaMemcpy( data.data(), d_data, sizeof( element_t ) * n, cudaMemcpyDeviceToHost );
    CHECK_ERROR( "Transferring result back to host" );
    cudaFree( d_data );
    if( algo == sort_algo::radix ) free_radix_scratch( scratch );
    CHECK_ERROR( "Freeing device memory" );

    auto const ns = std::chrono::duration_cast< std::chrono::nanoseconds >( kernel_end - kernel_start ).count();
    return { std::move( data ), static_cast< long long >( ns ) };
}

} // namespace gpu
} // namespace a1
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "main.cu"
/**
 * Driver for the benchmark comparison.
 *
 *  1. Runs the two examples from the assignment handout and prints them.
 *  2. For a range of sizes n, generates random data, runs the CPU baseline and
 *     all four GPU variants (bitonic / radix x global-only / shared memory),
 *     checks every GPU result against the CPU result, and prints the timings.
 */

#include <algorithm> // std::min
#include <cstddef>   // std::size_t type
#include <iomanip>   // std::setw
#include <iostream>  // std::cout, std::endl
#include <string>
#include <vector>

#include "algorithm_choices.h"
#include "data_generator.h"
#include "data_types.h"
#include "cuda_common.h"

namespace {

using namespace csc485b::a1;

struct variant
{
    char const *   name;
    gpu::sort_algo algo;
    gpu::mem_mode  mode;
};

variant const variants[] = {
    { "bitonic/global", gpu::sort_algo::bitonic, gpu::mem_mode::global_only },
    { "bitonic/shared", gpu::sort_algo::bitonic, gpu::mem_mode::shared      },
    { "radix/global",   gpu::sort_algo::radix,   gpu::mem_mode::global_only },
    { "radix/shared",   gpu::sort_algo::radix,   gpu::mem_mode::shared      },
};

void print( std::vector< element_t > const & v )
{
    std::cout << "[";
    for( std::size_t i = 0; i < v.size(); ++i ) std::cout << ( i ? ", " : "" ) << v[ i ];
    std::cout << "]";
}

/// Best-of-`reps` GPU time; also reports whether every run matched `expected`.
long long time_gpu( variant const & v, std::vector< element_t > const & data, std::size_t switch_at,
                    std::vector< element_t > const & expected, bool & all_correct, int reps )
{
    long long best = -1;
    for( int r = 0; r < reps; ++r )
    {
        auto const res = gpu::run_gpu_soln( data, switch_at, data.size(), v.algo, v.mode );
        all_correct = all_correct && ( res.output == expected );
        best = ( best < 0 ) ? res.ns : std::min( best, res.ns );
    }
    return best;
}

} // anonymous namespace

int main()
{
    // ---- 1. Handout examples ------------------------------------------------
    std::vector< std::vector< element_t > > const examples = {
        { 4, 7, 2, 9, 5, 8, 3, 1 },
        { 1, 2, 1, 2, 3, 4, 1, 5 },
    };

    std::cout << "== Handout examples ==\n";
    for( auto const & ex : examples )
    {
        std::size_t const n = ex.size();
        std::size_t const switch_at = 3 * ( n >> 2 );
        auto const cpu = cpu::run_cpu_baseline( ex, switch_at, n );

        std::cout << "input  "; print( ex );         std::cout << "\n";
        std::cout << "cpu    "; print( cpu.output ); std::cout << "\n";
        for( auto const & v : variants )
        {
            auto const res = gpu::run_gpu_soln( ex, switch_at, n, v.algo, v.mode );
            std::cout << std::left << std::setw( 15 ) << v.name << std::right;
            print( res.output );
            std::cout << ( res.output == cpu.output ? "  OK" : "  MISMATCH" ) << "\n";
        }
        std::cout << "\n";
    }

    // ---- 2. Benchmark -------------------------------------------------------
    std::vector< std::size_t > const sizes = {
        4, 100, 1000, 1024, 2048, 5000, 1 << 13, 1 << 14, 1 << 16,
        100000, 1 << 18, 1 << 20, 1 << 22
    };
    int const reps = 3;

    std::cout << "== Benchmark (ns, best of " << reps << " GPU runs; all GPU outputs checked against CPU) ==\n";
    std::cout << std::setw( 9 ) << "n" << std::setw( 14 ) << "CPU";
    for( auto const & v : variants ) std::cout << std::setw( 16 ) << v.name;
    std::cout << "  correct\n";

    bool everything_correct = true;
    for( std::size_t const n : sizes )
    {
        std::size_t const switch_at = 3 * ( n >> 2 );
        auto const data = generate_uniform< element_t >( n );
        auto const cpu  = cpu::run_cpu_baseline( data, switch_at, n );

        bool ok = true;
        std::cout << std::setw( 9 ) << n << std::setw( 14 ) << cpu.ns;
        for( auto const & v : variants )
            std::cout << std::setw( 16 ) << time_gpu( v, data, switch_at, cpu.output, ok, reps );
        std::cout << "  " << ( ok ? "yes" : "NO" ) << std::endl;
        everything_correct = everything_correct && ok;
    }

    std::cout << "\nAll GPU results match the CPU baseline: " << ( everything_correct ? "yes" : "NO" ) << std::endl;
    return everything_correct ? EXIT_SUCCESS : EXIT_FAILURE;
}

In [ ]:
%cuda_group_run --group "source" --compiler-args "-O0 -g -std=c++20 -arch=sm_75"

## Notes

**Opposing sort (bitonic).** The input is padded with `0xFFFFFFFF` to a power of two and run through a bitonic sorting network, whose last stage is all-ascending. One extra parallel kernel then reverses `[3n/4, n)`, which puts the largest n/4 elements in descending order. Since bitonic sort needs a grid-wide barrier between steps whose stride crosses block boundaries, the host launches one kernel per such step.

* **Global-only**: one `bitonic_step_global` launch per (k, j) step; every compare-exchange goes to DRAM.
* **Shared memory**: each 1024-thread block sorts a 2048-element tile completely in shared memory (all stages up to k = 2048). For larger stages, only strides j ≥ 2048 use global launches; strides 1024..1 run as one shared-memory launch per stage.

**Radix sort.** LSD, 4-bit digits, 8 passes: per-tile histogram, global exclusive scan of the histograms (digit-major), stable scatter, ping-pong buffers.

* **Global-only**: each thread histograms / scatters its own 32-element chunk serially (registers only); the scan is Hillis–Steele in global memory.
* **Shared memory**: each block sorts its 1024-key tile by the digit in shared memory (four stable 1-bit splits via a block scan), so the scatter writes contiguous runs; the scan is a recursive shared-memory block scan.

`mem_mode::automatic` (the default) uses the shared-memory path when n ≥ 1024. The benchmark forces each mode so they can be compared at every n.

**Timing.** Kernel launches are asynchronous, so the starter's timer only measured launch overhead. The timed region now ends with `cudaDeviceSynchronize()`. Setup (malloc, memcpy, padding) is still excluded.

**Warm-up caveat.** `warm_the_gpu` (starter code) increments any value above 100·n. It's harmless for the provided generator (range [0, 2n]) but would corrupt arbitrary 32-bit inputs.